In [3]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score
# Thư viện quan trọng để xử lý mất cân bằng
from imblearn.over_sampling import SMOTE
import warnings

warnings.filterwarnings('ignore')

In [4]:
def evaluate_model(y_true, y_pred, model_name, dataset_name):
    print(f"\n{'='*60}")
    print(f"{model_name} - {dataset_name}")
    print(f"{'='*60}")

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"- Accuracy:  {acc:.4f}")
    print(f"- Precision: {prec:.4f}")
    print(f"- Recall:    {rec:.4f}")
    print(f"- F1-Score:  {f1:.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=['Không bị', 'Bị']))

In [5]:
try:
    df3 = pd.read_csv("../../datasets/processed/diabetes_dataset3_scaled.csv")
except FileNotFoundError:
    print("Lỗi: Không tìm thấy file. Hãy chắc chắn đường dẫn `../../datasets/processed/diabetes_dataset3_scaled.csv` là chính xác.")
    # Xử lý lỗi nếu cần
    exit()


X = df3.drop('Outcome', axis=1)
y = df3['Outcome']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Kích thước tập train GỐC: {X_train.shape[0]} mẫu")
print(f"Kích thước tập test GỐC:  {X_test.shape[0]} mẫu")
print(f"Tỷ lệ 'Bị bệnh' (1) trong tập train GỐC: {y_train.mean():.2f}")

Kích thước tập train GỐC: 614 mẫu
Kích thước tập test GỐC:  154 mẫu
Tỷ lệ 'Bị bệnh' (1) trong tập train GỐC: 0.35


In [7]:
# ================================================
# BƯỚC MỚI: ÁP DỤNG SMOTE CHỈ TRÊN TẬP TRAIN
# ================================================
print("\nĐang áp dụng SMOTE để cân bằng tập train...")
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Kích thước tập train MỚI (đã SMOTE): {X_train_resampled.shape[0]} mẫu")
print(f"Tỷ lệ 'Bị bệnh' (1) trong tập train MỚI: {y_train_resampled.mean():.2f}")


Đang áp dụng SMOTE để cân bằng tập train...
Kích thước tập train MỚI (đã SMOTE): 802 mẫu
Tỷ lệ 'Bị bệnh' (1) trong tập train MỚI: 0.50


In [8]:
# ========================================================
# TRAIN KNN (VỚI K=19) TRÊN DỮ LIỆU ĐÃ CÂN BẰNG (SMOTE)
# ========================================================
# Sử dụng K=19 là K tốt nhất bạn tìm được ở file 03c
best_k = 19
knn_smote = KNeighborsClassifier(n_neighbors=best_k)

# Huấn luyện trên dữ liệu đã resample
knn_smote.fit(X_train_resampled, y_train_resampled)

# Đánh giá trên tập TEST GỐC (quan trọng: không dùng test đã smote)
y_pred = knn_smote.predict(X_test)

# In kết quả
evaluate_model(y_test, y_pred, f"KNN (K={best_k}) với SMOTE", "Dataset 3 (Scaled + SMOTE Train)")


KNN (K=19) với SMOTE - Dataset 3 (Scaled + SMOTE Train)
- Accuracy:  0.6818
- Precision: 0.5333
- Recall:    0.8727
- F1-Score:  0.6621

Confusion Matrix:
[[57 42]
 [ 7 48]]

Classification Report:
              precision    recall  f1-score   support

    Không bị       0.89      0.58      0.70        99
          Bị       0.53      0.87      0.66        55

    accuracy                           0.68       154
   macro avg       0.71      0.72      0.68       154
weighted avg       0.76      0.68      0.69       154



In [9]:
# ===========================
# LƯU MODEL
# ===========================
joblib.dump(knn_smote, "../../models/diabetes_knn_smote_model.pkl")
print("\n✅ Mô hình KNN-SMOTE đã được lưu thành công!")


✅ Mô hình KNN-SMOTE đã được lưu thành công!
